<a href="https://colab.research.google.com/github/mlft27/Hybrid-Search-and-Retrieval/blob/main/Hybrid_Search_%26_Retrieval_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Hybrid Search & Retrieval — Assignment**

## **Problem Statement**

Build a **hybrid search & retrieval system** over biomedical research abstracts using the [PubMedQA](https://huggingface.co/datasets/qiaojin/PubMedQA) dataset. Your system must combine **two complementary retrieval strategies** — dense (semantic) and sparse (lexical) — and fuse their results using **Reciprocal Rank Fusion (RRF)** so that the final ranking outperforms either method alone.

You will:

1. Load & clean a subset of PubMedQA  
2. Construct rich, searchable document text from structured fields  
3. Generate **dense** embeddings and **sparse** embeddings
4. Index documents in **Qdrant** with both vector types  
5. Implement a **hybrid search** function that queries both indexes in parallel  
6. Implement **RRF** to merge and re-rank the two result lists  
7. Test and compare all three retrieval modes (dense-only, sparse-only, hybrid)

### Architecture

```
                        ┌─── Dense (BGE-Large) ──────┐
Query ──► Embed Query ──┤                            ├──► RRF Fusion ──► Final Ranked Results         
                        └─── Sparse (SPLADE) ────────┘
```

### Key Concepts

- **Dense retrieval** excels at *semantic* / conceptual matching (e.g., "physical activity" ↔ "exercise")  
- **Sparse retrieval** excels at *exact* / lexical matching (e.g., drug names like "metformin")  
- **Hybrid search** combines both to cover each method's blind spots  
- **RRF** merges ranked lists without needing score normalization — a document found by *both* methods is promoted



## **💡 Tips**




**Document construction matters.** Think carefully about *which* fields to include in the document text and *which* to exclude. The `question` field is what users will search with — including it in the document would allow trivial self-matching and inflate your results.



## **Steps**

### Step 1: Setup & Installation
Install the required packages: `qdrant-client`, `fastembed`, `datasets`, `transformers`. Import all necessary modules.

You will need:
- `QdrantClient` and models from `qdrant_client.models` (`Distance`, `SparseVector`, `PointStruct`, `QueryRequest`, `SparseIndexParams`, `SparseVectorParams`, `VectorParams`, `ScoredPoint`)
- `SparseTextEmbedding` and `TextEmbedding` from `fastembed`
- `load_dataset` from `datasets`
- `pandas`, `numpy`, `json`


In [28]:
# Write your code here
!pip install -qU qdrant-client fastembed datasets transformers

In [5]:
import json

import numpy as np
import pandas as pd
from datasets import load_dataset
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    SparseVector,
    PointStruct,
    QueryRequest,
    SparseIndexParams,
    SparseVectorParams,
    VectorParams,
    ScoredPoint,
)
from transformers import AutoTokenizer

import fastembed
from fastembed import SparseEmbedding, SparseTextEmbedding, TextEmbedding

print(f"FastEmbed version: {fastembed.__version__}")

FastEmbed version: 0.9.0


### Step 2: Load & Explore the PubMedQA Dataset
Load the `pqa_artificial` split of `qiaojin/PubMedQA`. Print total instances and feature names. Then select a **subset of 100** instances to work with and convert to a pandas DataFrame.


In [6]:
# Write your code here
dataset = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

In [7]:
print(f"Total instances: {len(dataset)}")
print(f"Feature names: {dataset.column_names}")

Total instances: 211269
Feature names: ['pubid', 'question', 'context', 'long_answer', 'final_decision']


In [8]:
# We'll select the first 100 examples for this demo
dataset = dataset.select(range(100))
print(f"Working with {len(dataset)} instances")

Working with 100 instances


In [9]:
source_df = dataset.to_pandas() # convert a dataset object into a pandas DataFrame

### Step 3: Data Cleaning & Exploration
Before building documents, inspect the data:
- Check for duplicate `pubid` values  
- Check for missing values and empty strings in `question` and `long_answer`  
- Print the distribution of `final_decision` (yes / no / maybe)

This step ensures you understand what you're working with.


In [10]:
# Write your code here
# Deduplicate products (same product can appear with different queries)
df = source_df.drop_duplicates(
    subset=["pubid"]
)

In [11]:
# Drop rows with missing essential fields
df = df.dropna(subset=["question", "long_answer"])

In [12]:
print(f"\n--- Final Decision Distribution ---")
print(df['final_decision'].value_counts())


--- Final Decision Distribution ---
final_decision
yes    89
no     11
Name: count, dtype: int64


### Step 4: Construct the Document Text
Write a function that combines:
1. **Context paragraphs** — prefixed with their section label, e.g. `[BACKGROUND] ...`  
2. **Long answer** (conclusion) — prefixed with `[CONCLUSION]`  

**Important:** Do **NOT** include the `question` field in the document text — that's what users will query with.

Apply this function to create a new `combined_text` column in your DataFrame.


In [13]:
df.head()

,pubid,question,context,long_answer,final_decision
0,25429730,Are group 2 innate lymphoid cells ( ILC2s ) in...,{'contexts': ['Chronic rhinosinusitis (CRS) is...,"As ILC2s are elevated in patients with CRSwNP,...",yes
1,25433161,Does vagus nerve contribute to the development...,{'contexts': ['Phosphatidylethanolamine N-meth...,Neuronal signals via the hepatic vagus nerve c...,yes
2,25445714,Does psammaplin A induce Sirtuin 1-dependent a...,{'contexts': ['Psammaplin A (PsA) is a natural...,PsA significantly inhibited MCF-7/adr cells pr...,yes
3,25431941,Is methylation of the FGFR2 gene associated wi...,{'contexts': ['This study examined links betwe...,We identified a novel biologically plausible c...,yes
4,25432519,Do tumor-infiltrating immune cell profiles and...,{'contexts': ['Tumor microenvironment immunity...,Breast cancer immune cell subpopulation profil...,yes


In [14]:
# Write your code here
def construct_document_text(row) -> str:
    parts = []
    context = row["context"]

    # Ensure context is parsed correctly if it's a JSON string
    if isinstance(context, str):
        import json
        context = json.loads(context)

    # Combine context paragraphs with their corresponding section labels
    for para, label in zip(context.get("contexts", []), context.get("labels", [])):
        parts.append(f"[{label}] {para}")

    # Append the long answer with [CONCLUSION] prefix
    parts.append(f"[CONCLUSION] {row['long_answer']}")

    return "\n\n".join(parts)

# Create the combined_text column in df
df["combined_text"] = df.apply(construct_document_text, axis=1)

# Display the first few entries
print(f"Total documents: {len(df)}")
print("\nSample document content (first 500 chars):")
print(df["combined_text"].iloc[0][:500] + "...")

Total documents: 100

Sample document content (first 500 chars):
[BACKGROUND] Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.

[OBJECTIVE] The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of diseas...


### Step 5: Load Embedding Models
Initialize two models from FastEmbed:
- **Sparse model:** `prithvida/Splade_PP_en_v1` — using `SparseTextEmbedding`
- **Dense model:** `BAAI/bge-large-en-v1.5` — using `TextEmbedding`

Write helper functions `make_sparse_embedding(texts)` and `make_dense_embedding(texts)` that wrap each model's `.embed()` method.


In [15]:
# Write your code here
# Models we will use
sparse_model_name = "prithvida/Splade_PP_en_v1"
dense_model_name = "BAAI/bge-small-en-v1.5"

sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=16)
dense_model = TextEmbedding(model_name=dense_model_name, batch_size=16)

print("✅ Both models loaded!")

/tmp/ipykernel_19501/1056151872.py:6: DeprecationWarning: The right spelling is prithivida/Splade_PP_en_v1. Support of this name will be removed soon, please fix the model_name
  sparse_model = SparseTextEmbedding(model_name=sparse_model_name, batch_size=16)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

✅ Both models loaded!


In [16]:
# Helper functions
def make_sparse_embedding(texts: list[str]) -> list[SparseEmbedding]:
    return list(sparse_model.embed(texts, batch_size=16))

def make_dense_embedding(texts: list[str]):
    return list(dense_model.embed(texts))

### Step 6: Generate Embeddings
Embed all  documents using both models. Store results as new columns `sparse_embedding` and `dense_embedding` in the DataFrame.



In [17]:
# Write your code here
# df["sparse_embedding"] = df["combined_text"].apply(make_sparse_embedding)
# df["dense_embedding"] = df["combined_text"].apply(make_dense_embedding)
document_texts = df["combined_text"].tolist()
print(f"Embedding {len(document_texts)} products...")

Embedding 100 products...


In [18]:
%%time
print("⏳ Generating sparse embeddings (SPLADE)...")
df["sparse_embedding"] = make_sparse_embedding(document_texts)
print("✅ Done!")

⏳ Generating sparse embeddings (SPLADE)...
✅ Done!
CPU times: user 4min 45s, sys: 11.6 s, total: 4min 57s
Wall time: 4min 59s


In [19]:
%%time
print("⏳ Generating dense embeddings (BGE-Large)...")
df["dense_embedding"] = make_dense_embedding(document_texts)
print("✅ Done!")

⏳ Generating dense embeddings (BGE-Large)...
✅ Done!
CPU times: user 1min 13s, sys: 1.59 s, total: 1min 14s
Wall time: 1min 15s


### Step 7: Create the Qdrant Collection
Initialize an in-memory `QdrantClient`. Create a collection named `"pubmedqa"` configured with:
- A **dense vector** config named `"text-dense"` — size 1024, cosine distance  
- A **sparse vector** config named `"text-sparse"`


In [21]:
# Write your code here
client = QdrantClient(":memory:")

collection_name = "pubmedqa"

client.create_collection(
    collection_name,
    # Dense vector config: 1024-dim with cosine similarity
    vectors_config={
        "text-dense": VectorParams(
            size=384,             # Must match model output dimension
            distance=Distance.COSINE,  # Cosine similarity for dense vectors
        )
    },
    # Sparse vector config: no fixed size needed (it's dynamic)
    sparse_vectors_config={
        "text-sparse": SparseVectorParams(
            index=SparseIndexParams(
                on_disk=False,    # Keep in RAM for speed
            )
        )
    },
)

print(f"✅ Collection '{collection_name}' created with dense + sparse vector support")

✅ Collection 'pubmedqa' created with dense + sparse vector support


### Step 8: Prepare & Upload Points
Write a function `make_points(df)` that converts each DataFrame row into a `PointStruct` containing:
- **id:** row index  
- **payload:** `text`, `pubid`, `question`, `final_decision`  
- **vectors:** both `text-dense` (list) and `text-sparse` (`SparseVector` with indices and values)

Then upsert all points into the collection.


In [22]:
#Write your code here
def make_points(df: pd.DataFrame) -> list[PointStruct]:
    """Convert DataFrame rows into Qdrant PointStruct objects."""
    sparse_vectors = df["sparse_embedding"].tolist()
    medical_texts = df["combined_text"].tolist()
    dense_vectors = df["dense_embedding"].tolist()
    rows = df.to_dict(orient="records")

    points = []
    for idx, (text, sparse_vector, dense_vector) in enumerate(
        zip(medical_texts, sparse_vectors, dense_vectors)
    ):
        # Convert sparse embedding to Qdrant's SparseVector format
        sparse_vector = SparseVector(
            indices=sparse_vector.indices.tolist(),
            values=sparse_vector.values.tolist()
        )

        point = PointStruct(
            id=idx,
            payload={
                "text": text,
                "pubid": rows[idx]["pubid"],
                "question": rows[idx]["question"],
                "final_decision": rows[idx]["final_decision"]
            },
            vector={
                "text-sparse": sparse_vector,   # Sparse vector under its named key
                "text-dense": dense_vector.tolist(),  # Dense vector under its named key
            },
        )
        points.append(point)
    return points

points = make_points(df)
print(f"Prepared {len(points)} points for indexing")

Prepared 100 points for indexing


In [23]:
# Upload to Qdrant
client.upsert(collection_name, points)
print(f"✅ {len(points)} points indexed in Qdrant!")

✅ 100 points indexed in Qdrant!


### Step 9: Implement Hybrid Search
Write a `search(query_text, top_k=10)` function that:
1. Embeds the query with **both** models  
2. Runs dense and sparse queries in **parallel** using `client.query_batch_points()`  
3. Returns both result lists: `[dense_results, sparse_results]`


In [24]:
# Write your code here
def search(query_text: str, top_k: int = 10):
    """
    Perform hybrid search: run dense AND sparse search in parallel,
    return both result lists.
    """
    # Step 1: Embed the query with BOTH models
    query_sparse_vectors = make_sparse_embedding([query_text])
    query_dense_vector = make_dense_embedding([query_text])

    # Step 2: Fire two searches in a single batch request
    search_results = client.query_batch_points(
        collection_name=collection_name,
        requests=[
            # Search 1: Dense (semantic similarity)
            QueryRequest(
                query=query_dense_vector[0].tolist(),
                using="text-dense",
                limit=top_k,
                with_payload=True,
            ),
            # Search 2: Sparse (lexical matching)
            QueryRequest(
                query=SparseVector(
                    indices=query_sparse_vectors[0].indices.tolist(),
                    values=query_sparse_vectors[0].values.tolist(),
                ),
                using="text-sparse",
                limit=top_k,
                with_payload=True,
            ),
        ],
    )

    # Extract the point lists from QueryResponse objects
    return [search_results[0].points, search_results[1].points]






### Step 10: Test Search Queries
Test your search with **different query types** and compare dense vs. sparse results:

For each query, print the top 5 results from both dense and sparse, showing scores and text previews.


In [25]:
# Write your code here
# Query 1: Technical / specific — exact drug name
query_text = "Does metformin improve insulin sensitivity in type 2 diabetes?"
search_results = search(query_text)

dense_results, sparse_results = search_results[0], search_results[1]

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(dense_results[:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(sparse_results[:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

Query: 'Does metformin improve insulin sensitivity in type 2 diabetes?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.7101] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus and cognitive dysfunctio...
  2. [Score: 0.6695] [BACKGROUND] Plasma levels of pancreatic polypeptide (PP) rise upon food intake. Although other panc...
  3. [Score: 0.6646] [BACKGROUND] Recently, the role of osteoprotegerin (OPG) in the pathogenesis of heart failure throug...
  4. [Score: 0.6404] [BACKGROUND] Hyperglycemia on admission is associated with increased mortality rates in patients wit...
  5. [Score: 0.6288] [BACKGROUND] Stroke is the third leading cause of invalidism and death in industrialized countries. ...

SPARSE (Lexical) Results — Top 5:
  1. [Score: 8.8905] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus and cognitive dysfunctio...
  2. [Score: 7.9734] [BACKGROUND] Guillotine below-knee amputation (BKA) for wet gangrene is an unfortunate 

In [26]:
# Query 2: Conceptual / synonym-heavy
query_text = "Can physical activity help with mental health conditions?"
search_results_2 = search(query_text)

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_2[0][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_2[1][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

Query: 'Can physical activity help with mental health conditions?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.6983] [BACKGROUND] In recent years, the relationship between physical activity (PA) and smoking cessation ...
  2. [Score: 0.6695] [BACKGROUND] Active commuters have lower risk of chronic disease. Understanding which of the, to som...
  3. [Score: 0.6430] [BACKGROUND] The prevalence of obesity in children is high, and many do not meet physical activity r...
  4. [Score: 0.6271] [BACKGROUND] Disturbed sleep is associated with mood disorders. Both depression and insomnia may inc...
  5. [Score: 0.6132] [BACKGROUND] Normal aging is associated with a decline in cognitive abilities, particularly in the d...

SPARSE (Lexical) Results — Top 5:
  1. [Score: 10.6326] [BACKGROUND] In recent years, the relationship between physical activity (PA) and smoking cessation ...
  2. [Score: 7.6956] [BACKGROUND] Active commuters have lower risk of chronic disease. Understanding which of th

In [27]:
# Query 3: Mixed — specific term + conceptual outcome
query_text = "Is statin therapy associated with reduced mortality?"
search_results_3 = search(query_text)

print(f"Query: '{query_text}'")
print(f"\n{'='*70}")
print(f"DENSE (Semantic) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_3[0][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

print(f"\n{'='*70}")
print(f"SPARSE (Lexical) Results — Top 5:")
print(f"{'='*70}")
for i, point in enumerate(search_results_3[1][:5]):
    title = point.payload['text'][:100].replace('\n', ' ')
    print(f"  {i+1}. [Score: {point.score:.4f}] {title}...")

Query: 'Is statin therapy associated with reduced mortality?'

DENSE (Semantic) Results — Top 5:
  1. [Score: 0.6813] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus and cognitive dysfunctio...
  2. [Score: 0.6799] [BACKGROUND] Obese patients with idiopathic pulmonary fibrosis (IPF) have higher 90-day mortality af...
  3. [Score: 0.6748] [BACKGROUND] Recently, the role of osteoprotegerin (OPG) in the pathogenesis of heart failure throug...
  4. [Score: 0.6713] [BACKGROUND] The goal of this study was to identify genetic determinants of plasma N-terminal proatr...
  5. [Score: 0.6683] [BACKGROUND] Immune monitoring (IM) has not been shown to be associated with cardiac allograft vascu...

SPARSE (Lexical) Results — Top 5:
  1. [Score: 9.4858] [BACKGROUND] Hyperglycemia on admission is associated with increased mortality rates in patients wit...
  2. [Score: 8.4595] [BACKGROUND] Stroke is the third leading cause of invalidism and death in industrialized countrie

### Step 11: Implement Reciprocal Rank Fusion (RRF)
Implement two functions:

1. `rank_list(search_result)` — converts Qdrant `ScoredPoint` results into `(id, rank)` pairs (1-indexed)  
2. `rrf(rank_lists, k=60, default_rank=1000)` — takes multiple rank lists and returns a single fused list sorted by RRF score

**RRF formula:** For each item, sum `1 / (k + rank)` across all lists. Items not present in a list get `default_rank`.


In [29]:
# Write your code here
def rank_list(search_result: list[ScoredPoint]):
    """Convert Qdrant search results to (id, rank) pairs."""
    return [(point.id, rank + 1) for rank, point in enumerate(search_result)]

# Convert both result lists to rank lists
dense_rank_list = rank_list(search_results[0])
sparse_rank_list = rank_list(search_results[1])

print("Dense ranks:", dense_rank_list[:5], "...")
print("Sparse ranks:", sparse_rank_list[:5], "...")


Dense ranks: [(93, 1), (18, 2), (47, 3), (8, 4), (77, 5)] ...
Sparse ranks: [(93, 1), (88, 2), (77, 3), (18, 4), (47, 5)] ...


In [30]:
def rrf(rank_lists, alpha=60, default_rank=1000):
    """
    Reciprocal Rank Fusion (RRF).

    Takes multiple rank lists and produces a single fused ranking.
    Only considers RANK POSITION, not raw scores — making it safe to
    combine results from different scoring systems.

    Args:
        rank_lists: List of [(item_id, rank), ...] lists
        alpha: Smoothing constant (default=60, from Cormack et al. 2009)
        default_rank: Rank assigned to items not in a list (high = penalty)

    Returns:
        Sorted list of (item_id, rrf_score) tuples
    """
    all_items = set(item for rank_list in rank_lists for item, _ in rank_list)
    item_to_index = {item: idx for idx, item in enumerate(all_items)}

    # Matrix: rows = items, cols = rank lists, filled with default_rank
    rank_matrix = np.full((len(all_items), len(rank_lists)), default_rank)

    for list_idx, rank_list in enumerate(rank_lists):
        for item, rank in rank_list:
            rank_matrix[item_to_index[item], list_idx] = rank

    # RRF formula: sum of 1/(alpha + rank) across all lists
    rrf_scores = np.sum(1.0 / (alpha + rank_matrix), axis=1)

    sorted_indices = np.argsort(-rrf_scores)  # Descending
    sorted_items = [(list(item_to_index.keys())[idx], rrf_scores[idx]) for idx in sorted_indices]

    return sorted_items

### Step 12: Apply RRF & Analyze Results
Apply RRF to fuse dense and sparse results from one of your test queries. Print the final ranked list, showing for each result:
- Its rank  
- Whether it came from **BOTH**, **Dense only**, or **Sparse only**  
- A text preview

**Analyze:** Do you see documents promoted that appeared in both lists? Are there any results that only one method found?


In [31]:
# Write your code here
# Fuse the rankings!
rrf_rank_list = rrf([dense_rank_list, sparse_rank_list])

print(f"\n🏆 HYBRID SEARCH RESULTS for: '{query_text}'")
print(f"{'='*70}")

# Retrieve full records for the fused results
fused_records = client.retrieve(
    collection_name=collection_name,
    ids=[item[0] for item in rrf_rank_list]
)

# Create a lookup by ID
record_by_id = {r.id: r for r in fused_records}

for rank, (item_id, score) in enumerate(rrf_rank_list, 1):
    record = record_by_id[item_id]
    title = record.payload['text'].split('\n')[0][:75]

    # Check if this item was in dense, sparse, or both
    in_dense = any(id == item_id for id, _ in dense_rank_list)
    in_sparse = any(id == item_id for id, _ in sparse_rank_list)
    source = "BOTH" if (in_dense and in_sparse) else ("Dense only" if in_dense else "Sparse only")

    print(f"  {rank:>2}. [{source:<12}] {title}")


🏆 HYBRID SEARCH RESULTS for: 'Is statin therapy associated with reduced mortality?'
   1. [BOTH        ] [OBJECTIVE] To determine whether older adults with type 2 diabetes mellitus
   2. [BOTH        ] [BACKGROUND] Plasma levels of pancreatic polypeptide (PP) rise upon food in
   3. [BOTH        ] [BACKGROUND] Recently, the role of osteoprotegerin (OPG) in the pathogenesi
   4. [BOTH        ] [BACKGROUND] Stroke is the third leading cause of invalidism and death in i
   5. [BOTH        ] [BACKGROUND] The role of Kdr (VEGFR-2/Flk-1) in vascular formation has been
   6. [Sparse only ] [BACKGROUND] Guillotine below-knee amputation (BKA) for wet gangrene is an 
   7. [Dense only  ] [BACKGROUND] Hyperglycemia on admission is associated with increased mortal
   8. [Dense only  ] [OBJECTIVE] Phosphatidylethanolamine N-methyltransferase (PEMT), a liver en
   9. [Sparse only ] [BACKGROUND] Immune monitoring (IM) has not been shown to be associated wit
  10. [Dense only  ] [BACKGROUND] The goal